# DSE 200 — Day 1 Lab
## Python Essentials and NumPy

**Chunk 3 · 1:00 – 3:30 · agent off**

Section numbers match the slides. Slide 3.9 is section 3.9 here.

| Cell | What to do |
| --- | --- |
| `▶ DEMO` | Already written. Run it while Rick talks. |
| `🔍 READ` | **Predict the output before running.** Commit to an answer. |
| `✍️ YOU TRY` | Has a `# TODO`. Yours. |
| `✅ CHECKPOINT` | Run it. It tells you whether you are right. |

**Open the table of contents** — the ☰ icon on the left. It is your map for the next two and a half hours.

**If you get lost:** *Runtime → Run all*. Every section rebuilds its own data, so any section works from cold.

### The course rule

> **You must be able to explain every cell you submit.**

Not "you must have typed it". Explain it — what it does, what it assumes, where it breaks.

That is what the `🔍 READ` cells are for. Writing code is the easy half.

In [ ]:
# ▶ SETUP — run this first. It loads the data and defines check_read().
import os, csv, io, random, textwrap

CANDIDATES = ["temperatures.csv", "/content/temperatures.csv",
              "data/temperatures.csv", "../temperatures.csv"]
PATH = next((p for p in CANDIDATES if os.path.exists(p)), None)
if PATH is None:  # opened from GitHub: Colab brought the notebook but not the data
    try:
        import urllib.request
        urllib.request.urlretrieve("https://raw.githubusercontent.com/DSE200-2026/DSE200/main/temperatures.csv", "temperatures.csv")
        PATH = "temperatures.csv"
        print("downloaded temperatures.csv from the course repo")
    except Exception as e:
        print("could not download temperatures.csv:", e)

if PATH is None:
    # Fallback: rebuild a file with the same shape and the same defects.
    rng = random.Random(0)
    cities = ["San Diego", "Austin", "New York", "Phoenix", "Seattle"]
    rows = []
    for _ in range(5000):
        d = f"2023-{rng.randint(1,12):02d}-{rng.randint(1,28):02d}"
        rows.append([d, rng.choice(cities), f"{rng.uniform(0,40):.1f}"])
    for i in rng.sample(range(5000), 488): rows[i][2] = ""
    for i in rng.sample(range(5000), 470): rows[i][1] = ""
    for i in rng.sample(range(5000), 499): rows[i][0] = ""
    PATH = "temperatures.csv"
    with open(PATH, "w", newline="") as f:
        w = csv.writer(f); w.writerow(["date","city","temperature (C)"]); w.writerows(rows)
    print("NOTE: temperatures.csv not found — generated an equivalent file.")

print("data file:", PATH)

_READ = {
"3.2": ("0.30000000000000004\nFalse",
        "Floats are stored in binary, and 0.1 has no exact binary form -- just as 1/3\n"
        "has no exact decimal form. The tiny error survives the addition.\n"
        "Never compare computed floats with ==. Use a tolerance, or round first."),
"3.4": ("warm",
        "35 >= 20 is True, so the FIRST branch wins and the elif is never reached.\n"
        "The 'hot' branch is unreachable -- a bug that raises no error.\n"
        "Put the tightest condition first."),
"3.7": ("[1, 2, 3, 99]",
        "b = a did not copy anything. It put a SECOND LABEL on the same list.\n"
        "This is 'identity' from this morning: two names, one value in memory.\n"
        "Use a.copy(), a[:], or list(a) when you mean a copy."),
"3.9": ("not in the data",
        "The else on a for loop runs only if the loop finished WITHOUT hitting break.\n"
        "It is the tidy way to say 'and if we never found it'.\n"
        "You will rarely write this. You will read it."),
"3.16":("[False False False  True False  True  True]",
        "Seven answers, not one -- one per element, same shape as the array.\n"
        "That array of True/False is a MASK, and you can index with it:\n"
        "    temps[temps > 25]   ->  only the elements where it was True."),
}

def check_read(tag, prediction):
    """Reveal the answer to a READ exercise -- after you commit to one."""
    if not str(prediction).strip() or str(prediction).strip() in ("???", "?"):
        print("Write a prediction first. Committing to a wrong answer is how this works.")
        return
    ans, why = _READ[tag]
    print("your prediction:"); print(textwrap.indent(str(prediction), "    "))
    print("\nactual:");        print(textwrap.indent(ans, "    "))
    print("\n" + why)

print("check_read() ready.")

## 3.1 The Data

5,000 rows. Every day of 2023, five cities.

**The question:** what is the average temperature in Seattle?

You cannot answer that by looking. Nor these: how many readings are missing? Which city swings widest? Is Phoenix hotter than Austin?

By 3:30 you can answer all of them — and one of the answers will surprise you.

In [ ]:
# ▶ DEMO — look at the raw file first. Always.
with open(PATH) as f:
    for i, line in enumerate(f):
        print(repr(line))
        if i == 6: break

In [ ]:
# ▶ DEMO — how big is it, and what is missing?
with open(PATH, newline="") as f:
    rows = list(csv.DictReader(f))

print("rows:", len(rows))
print("columns:", list(rows[0]))
print("blank temperature:", sum(1 for r in rows if not r["temperature (C)"].strip()))
print("blank city       :", sum(1 for r in rows if not r["city"].strip()))
print("blank date       :", sum(1 for r in rows if not r["date"].strip()))

## 3.2 Values and Types

A variable is a **label you put on a value**. You never say what kind of thing it is — Python works that out, and you can move the label somewhere else at any time.

> This morning's word for this is **binding**: the association between the name of a thing and its value.

In [ ]:
# ▶ DEMO — five types, and moving a label
city       = "Seattle"      # str
reading    = 8.4            # float
n_readings = 937            # int
is_hot     = False          # bool
best       = None           # NoneType

for v in (city, reading, n_readings, is_hot, best):
    print(f"{repr(v):>10}  ->  {type(v).__name__}")

x = 3
x = 4.5
x = "warm"          # nothing complains
print("\nx is now", type(x).__name__)

In [ ]:
# ▶ DEMO — converting is something you ask for
print(int("42") + 1)
print(float("8.4") * 2)
print(int(3.99), "<- chops", round(3.99), "<- rounds")

try:
    int("12abc")
except ValueError as e:
    print("ValueError:", e)      # remember this. It is why section 3.12 is hard.

In [ ]:
# ▶ DEMO — operators, and f-strings
a, b = 17, 5
print(a / b, a // b, a % b, a ** 2)

city, n, share = "Seattle", 937, 0.0976
print(f"{city} has {n:,} readings")
print(f"{share:.1%} are missing")
print(f"{8.4361:.2f}   {42:05d}")

In [ ]:
# 🔍 READ 3.2 — what does this print? Two lines.
#
#     print(0.1 + 0.2)
#     print(0.1 + 0.2 == 0.3)
#
# Commit to an answer, then run this cell.

prediction = "???"
check_read("3.2", prediction)

In [ ]:
# ▶ DEMO — so this is how you compare floats
print(abs((0.1 + 0.2) - 0.3) < 1e-9)
print(round(0.1 + 0.2, 10) == 0.3)

In [ ]:
# ✍️ YOU TRY 3.2
# A model was tested on 1,284 readings and got 1,197 right.

n_total, n_correct = 1284, 1197

accuracy = 0   # TODO: a fraction between 0 and 1
n_wrong  = 0   # TODO: a whole number

print(f"accuracy={accuracy:.4f}  wrong={n_wrong}")

In [ ]:
# ✅ CHECKPOINT 3.2
assert abs(accuracy - 0.93224) < 1e-4, f"accuracy is {accuracy}"
assert n_wrong == 87, f"n_wrong is {n_wrong}"
print("Correct.")

## 3.3 Logic

Comparisons give you `True` or `False`. Combine them with `and`, `or`, `not` — English words, not `&&` and `||`.

In [ ]:
# ▶ DEMO
temp, humidity = 31.5, 0.82

print(temp > 30, temp == 31.5, 20 <= temp <= 35)     # chained, reads like maths
print(temp > 30 and humidity > 0.8)
print(temp < 0 or humidity > 0.8)
print(not (temp > 30))
print("Sea" in "Seattle")

In [ ]:
# ▶ DEMO — truthiness. These are ALL falsy.
for v in [False, 0, 0.0, "", [], {}, (), None, "no", 42]:
    print(f"{repr(v):>7}  ->  {bool(v)}")

# which is why "does this row have a temperature?" is just:
value = ""
print("\nhas a reading:", bool(value))

In [ ]:
# ▶ DEMO — is vs ==
a = [1, 2, 3]
b = [1, 2, 3]
print(a == b, a is b)      # equal value, different objects

reading = None
print(reading is None)     # the ONLY place you should use `is` today

## 3.4 Branching

**Indentation is the syntax.** Four spaces. Only the *first* matching branch runs — so order your conditions from tightest to loosest.

In [ ]:
# ▶ DEMO
temp = 31.5

if temp >= 30:
    label = "hot"
elif temp >= 20:
    label = "warm"
elif temp >= 10:
    label = "mild"
else:
    label = "cold"

print(temp, "->", label)
print("one-liner:", "hot" if temp >= 30 else "not hot")

In [ ]:
# 🔍 READ 3.4 — what does this print?
#
#     temp = 35
#     if temp >= 20:
#         print("warm")
#     elif temp >= 30:
#         print("hot")
#
# Commit first.

prediction = "???"
check_read("3.4", prediction)

In [ ]:
# ✍️ YOU TRY 3.4 — classify a temperature in Celsius.
#   below 10 -> "cold"   10-19 -> "mild"   20-29 -> "warm"   30+ -> "hot"

temp = 22.8

category = "?"   # TODO: an if / elif / elif / else

print(temp, "->", category)

In [ ]:
# ✅ CHECKPOINT 3.4
def _ref(t):
    return "hot" if t >= 30 else "warm" if t >= 20 else "mild" if t >= 10 else "cold"

assert category == _ref(temp), f"for {temp} you gave {category!r}, expected {_ref(temp)!r}"
print(f"Correct for temp = {temp}.")
print("Now change `temp` in the cell above and re-run until you have seen all four labels.")
print("Boundaries worth trying:", [9.9, 10.0, 19.9, 20.0, 29.9, 30.0])

## 3.5 When It Breaks

A traceback is read **bottom up**. The last line says what went wrong; the lines above say where.

| Error | Usually means |
| --- | --- |
| `SyntaxError` | typo — missing `:`, `)`, quote. Check the line *above*. |
| `NameError` | typo, or you never ran the cell that defines it |
| `TypeError` | wrong kind of thing — text where a number belongs |
| `ValueError` | right type, impossible value — `int("12abc")` |
| `KeyError` / `IndexError` | asked for something that is not there |

In [ ]:
# ▶ DEMO — meet them all at once
for snippet in ['1 + "a"', 'undefined_name', '[1,2,3][9]', '{"a":1}["b"]', 'int("x")']:
    try:
        eval(snippet)
    except Exception as e:
        print(f"{snippet:<16} -> {type(e).__name__}: {e}")

In [ ]:
# ▶ DEMO — the habit: print what you assumed
def peek(obj, name="obj"):
    """Type, size, and a sample of anything. Paste this into every notebook."""
    print(f"{name}: type={type(obj).__name__}", end=" ")
    try:
        print(f"len={len(obj)}", end=" ")
    except TypeError:
        pass
    try:
        print(f"first={list(obj)[0]!r}")
    except (TypeError, IndexError):
        print(f"value={obj!r}")

peek(rows, "rows")
peek(3.14, "pi")

## 3.6 Lists

Ordered, changeable, indexed from **zero**. Negative positions count from the right.

```
  23.7   8.4  11.2  31.5  19.0
    0     1     2     3     4
   -5    -4    -3    -2    -1
```

In [ ]:
# ▶ DEMO
temps  = [23.7, 8.4, 11.2, 31.5, 19.0, 27.3, 14.8]
cities = ["San Diego", "Seattle", "New York", "Phoenix", "Austin"]

print(len(temps), temps[0], temps[-1])
print(min(temps), max(temps), round(sum(temps)/len(temps), 2))
print(23.7 in temps, temps.index(11.2), temps.count(8.4))

In [ ]:
# ▶ DEMO — lists can be changed
r = [3, 1, 2]
r.append(9); r.insert(0, 7); r.extend([4, 4])
print(r)
r.remove(4); last = r.pop(); r[0] = 100
print(r, "popped:", last)

In [ ]:
# ▶ DEMO — slicing [start:stop:step]
nums = [0,1,2,3,4,5,6,7,8,9]
print(nums[2:5], nums[:3], nums[-3:])
print(nums[::2], nums[::-1])
print(nums[5:999], "<- a slice clamps")
try:
    nums[999]
except IndexError as e:
    print("IndexError:", e, "<- an index does not")

In [ ]:
# ▶ DEMO — the .sort() trap
nums = [5, 2, 9, 1]
print(sorted(nums), "original untouched:", nums)

result = nums.sort()
print("after .sort():", nums, "-- returned:", result)   # None!

In [ ]:
# ✍️ YOU TRY 3.6
readings = [12, 45, 7, 88, 23, 56, 91, 3, 67]

last_three  = []   # TODO the final three
every_other = []   # TODO positions 0, 2, 4, ...
backwards   = []   # TODO reversed, WITHOUT changing readings
ranked      = []   # TODO sorted largest first

print(last_three, every_other, backwards, ranked, sep="\n")
print("unchanged:", readings)

In [ ]:
# ✅ CHECKPOINT 3.6
assert last_three == [91, 3, 67], last_three
assert every_other == [12, 7, 23, 91, 67], every_other
assert backwards[0] == 67 and backwards[-1] == 12, backwards
assert ranked[:3] == [91, 88, 67], ranked[:3]
assert readings[0] == 12, "you mutated the original -- use sorted(), not .sort()"
print("Correct.")

## 3.7 Copies and Aliases

> **Identity**, from this morning. Two distinct names can point at *one* value in memory.

In [ ]:
# 🔍 READ 3.7 — what does this print?
#
#     a = [1, 2, 3]
#     b = a
#     a.append(99)
#     print(b)
#
# Commit first.

prediction = "???"
check_read("3.7", prediction)

In [ ]:
# ▶ DEMO — prove it, then fix it
a = [1, 2, 3]
b = a              # alias -- same object
c = a.copy()       # copy  -- or a[:] or list(a)

a.append(99)
print("a:", a)
print("b:", b, "<- changed too")
print("c:", c, "<- independent")
print(a is b, a is c)

## 3.8 Strings

A string is a sequence too — but **immutable**. Every method hands back a *new* string.

In [ ]:
# ▶ DEMO
s = "Seattle"
print(s[0], s[0:3], s[::-1], len(s))
try:
    s[0] = "B"
except TypeError as e:
    print("TypeError:", e)

print(s.upper(), "| original:", s)

In [ ]:
# ▶ DEMO — the methods you will actually use
raw = "  Seattle  "
print(repr(raw.strip()), raw.strip().lower())
print("a,b,c".split(","))
print("-".join(["a", "b", "c"]))
print("8.4".isdigit(), "<- the dot is not a digit")

In [ ]:
# ▶ DEMO — one row of our file, taken apart
line = " 2023-11-02 , Seattle , 8.4 \n"

fields = line.strip().split(",")
print(fields, "<- still spacey")

date, city, temp = [f.strip() for f in fields]
print(date, city, float(temp) + 1)

In [ ]:
# ✍️ YOU TRY 3.8
# This is a real row from the file -- note the empty temperature.
line = " 2023-05-19 ,Austin,  \n"

clean = []   # TODO: ['2023-05-19', 'Austin', '']

print(clean)
print("has a reading:", bool(clean[2]))

In [ ]:
# ✅ CHECKPOINT 3.8
assert clean == ["2023-05-19", "Austin", ""], clean
assert bool(clean[2]) is False, "an empty string is falsy -- that is the missing-value check"
print("Correct.")

## 3.9 Iteration

One idea — visit each item in turn — in six shapes. What changes is what you get handed.

| You want | Use |
| --- | --- |
| each item | `for x in things` |
| a count of turns | `for i in range(n)` |
| item **and** position | `for i, x in enumerate(things)` |
| two sequences in step | `for a, b in zip(xs, ys)` |
| keep going until a condition | `while` |
| stop early / skip one | `break` / `continue` |

In [ ]:
# ▶ DEMO — for, and range
for t in temps[:3]:
    print(t)

print(list(range(5)), list(range(2, 8)), list(range(0, 10, 3)), list(range(10, 0, -2)))

In [ ]:
# ▶ DEMO — the accumulator pattern. Learn this shape.
total = 0
hot = []

for t in temps:
    total += t
    if t > 25:
        hot.append(t)

print(round(total / len(temps), 2), hot)

In [ ]:
# ▶ DEMO — enumerate: the item AND where it is
for i, t in enumerate(temps[:3]):
    print(i, t)

for rank, city in enumerate(cities[:3], start=1):
    print(f"{rank}. {city}")

# you almost never want this:
for i in range(len(temps[:3])):
    print(i, temps[i], "<- longer, and it can go out of range")

In [ ]:
# ▶ DEMO — zip: two sequences in step
for city, t in zip(cities, temps):
    print(f"{city}: {t}")

print(list(zip([1,2,3], "abc")))

In [ ]:
# ▶ DEMO — unpacking in the loop header
# Each turn hands you a PAIR, and the two names on the left split it apart.
pair = ("Seattle", 8.4)
city, temp = pair
print(city, temp)

pairs = [("Seattle", 8.4), ("Austin", 19.0), ("Phoenix", 31.5)]
for city, temp in pairs:
    print(city, temp)

In [ ]:
# ▶ DEMO — while, break, continue
secret, guess, tries = 7, 0, 0
while guess != secret:              # you cannot know how many turns this takes
    guess += 1; tries += 1
print("found in", tries, "tries")

for t in temps:
    if t > 25:
        print("first hot reading:", t); break

clean = []
for raw in ["12", "", "45", "", "7"]:
    if not raw:
        continue                    # our missing-data problem, in four lines
    clean.append(float(raw))
print(clean)

In [ ]:
# 🔍 READ 3.9 — there is an `else` attached to the `for`.
#
#     target = 99
#     for t in temps:
#         if t == target:
#             print("found"); break
#     else:
#         print("not in the data")
#
# When does that else run? What prints?  Commit first.

prediction = "???"
check_read("3.9", prediction)

In [ ]:
# ▶ DEMO — and the version you will actually write
target = 99
print(any(t == target for t in temps))
print(all(t > 0 for t in temps))
print(sum(1 for t in temps if t > 25), "readings above 25")

In [ ]:
# ✍️ YOU TRY 3.9
# 1. first_hot_index -- POSITION of the first reading above 25, or -1 if none.
# 2. deltas          -- day-over-day change, rounded to 1 dp. One fewer element.

first_hot_index = -1   # TODO
deltas = []            # TODO

print(first_hot_index)
print(deltas)

In [ ]:
# ✅ CHECKPOINT 3.9
assert first_hot_index == 3, f"got {first_hot_index} -- temps[3] is 31.5"
assert len(deltas) == len(temps) - 1, f"expected {len(temps)-1} deltas, got {len(deltas)}"
assert deltas[0] == -15.3, deltas[0]
assert deltas[-1] == -12.5, deltas[-1]
print("Correct.")

## 3.10 Dictionaries

Lists answer *"what is at position 3?"*. Our questions are about a **name** — what is the average in Seattle?

In [ ]:
# ▶ DEMO
station = {"city": "Seattle", "readings": 937, "mean_temp": 19.3}

print(station["city"], len(station), "city" in station)
station["country"] = "USA"          # assigning a NEW key creates it
station["citty"]   = "typo"         # ...including a typo. No error.
print(station)
del station["citty"]

In [ ]:
# ▶ DEMO — [] raises, .get() does not
try:
    station["elevation"]
except KeyError as e:
    print("KeyError:", e)

print(station.get("elevation"))          # None
print(station.get("elevation", 0))       # your own default

In [ ]:
# ▶ DEMO — iterating, and the trap
for key, value in station.items():
    print(f"{key:>10}: {value}")

# do NOT change it while you walk it
try:
    for k in station:
        station.pop(k)
except RuntimeError as e:
    print("\nRuntimeError:", e)

## 3.11 The Two Idioms

Counting and grouping. Learn both cold — they are most of what you will do to data for the rest of your career.

In [ ]:
# ▶ DEMO — counting.  d[k] = d.get(k, 0) + 1
sample = ["Seattle", "Austin", "Seattle", "Phoenix", "Austin", "Seattle"]

counts = {}
for c in sample:
    counts[c] = counts.get(c, 0) + 1
print(counts)

for city, n in sorted(counts.items(), key=lambda p: p[1], reverse=True):
    print(f"{city:>10} {n}  {'#' * n}")

In [ ]:
# ▶ DEMO — grouping.  d.setdefault(k, []).append(v)
pairs = [("Seattle", 8.4), ("Austin", 19.0), ("Seattle", 11.2),
         ("Phoenix", 31.5), ("Austin", 22.6), ("Seattle", 9.1)]

by_city = {}
for city, temp in pairs:
    by_city.setdefault(city, []).append(temp)
print(by_city)

averages = {c: round(sum(v)/len(v), 1) for c, v in by_city.items()}
print(averages)

# collections.defaultdict(list) does the same thing automatically -- you will see both.

**Split, apply, combine.** Split the rows into groups, apply a calculation to each, combine the answers.

Tomorrow this entire block becomes `df.groupby("city")["temperature"].mean()`. You are writing it by hand today so that tomorrow you know what it is doing — and so that when an agent writes it for you, you can tell whether it grouped by the right thing.

## 3.12 Practice — 8 minutes

Answer the Seattle question yourself, on all 5,000 rows. Everything you need is `for`, `if`, `float()`, and a dictionary.

In [ ]:
# ▶ SETUP 3.12 — rebuilds from the file, so this section works from cold.
with open(PATH, newline="") as f:
    rows = list(csv.DictReader(f))

dates      = [r["date"].strip()            for r in rows]
city_col   = [r["city"].strip()            for r in rows]
temps_text = [r["temperature (C)"].strip() for r in rows]

print(len(rows), "rows loaded")
print("first three:", list(zip(dates, city_col, temps_text))[:3])

In [ ]:
# ✍️ YOU TRY 3.12
# 1. n_missing  -- how many rows have a BLANK temperature
# 2. by_city    -- city -> list of its usable temperatures (skip blank city OR blank temp)
# 3. averages   -- city -> mean temperature, rounded to 1 dp
# 4. print a ranked table, warmest first

n_missing = 0   # TODO
by_city   = {}  # TODO
averages  = {}  # TODO

# TODO: ranked table

In [ ]:
# ✅ CHECKPOINT 3.12
assert n_missing == 488, f"expected 488 blanks, got {n_missing}"
assert set(by_city) == {"San Diego","Austin","New York","Phoenix","Seattle"}, sorted(by_city)
assert sum(len(v) for v in by_city.values()) == 4090, sum(len(v) for v in by_city.values())
assert all(isinstance(v, float) for v in by_city["Seattle"]), "convert to float"
assert abs(averages["Seattle"] - 19.3) < 0.15, averages["Seattle"]
print("Correct. You answered the Seattle question.")
print("Hold on to `averages` -- we come back to it at 3.18.")

## 3.13 Sets

Unordered, no duplicates, instant membership.

In [ ]:
# ▶ DEMO
unique = set(city_col)
print(sorted(unique), len(unique))
print("Seattle" in unique)

print("\nrows:", len(city_col), " distinct cities:", len(unique - {""}))

In [ ]:
# ▶ DEMO — set algebra
train = {"Seattle", "Austin", "Phoenix", "New York"}
test  = {"Phoenix", "New York", "San Diego"}

print("in both  :", train & test)
print("all      :", train | test)
print("train only:", train - test)
print("exactly one:", train ^ test)

> `train & test` is the **data leakage** check. If your training data and your test data share rows, your model looks brilliant and is worthless — it has already seen the answers.
>
> **Thursday afternoon you will hunt a planted leak in an agent's notebook.** This is the line you will reach for.

In [ ]:
# ✍️ YOU TRY 3.13
v1_raw = ["s01","s02","s03","s04","s02","s05","s01","s06"]
v2_raw = ["s04","s05","s06","s07","s08","s05"]

only_v1    = set()   # TODO ids in v1 but not v2
only_v2    = set()   # TODO ids in v2 but not v1
both       = set()   # TODO in both -- the leak
n_dupes_v1 = 0       # TODO how many duplicate entries v1_raw contains

print(sorted(only_v1), sorted(only_v2), sorted(both), n_dupes_v1)

In [ ]:
# ✅ CHECKPOINT 3.13
assert only_v1 == {"s01","s02","s03"}, only_v1
assert only_v2 == {"s07","s08"}, only_v2
assert both == {"s04","s05","s06"}, both
assert n_dupes_v1 == 2, n_dupes_v1
print("Correct.")

## 3.14 Tuples

Like a list, but **frozen**. Which is exactly why it can be a dictionary key.

> **Mutability**, from this morning — whether the data may be changed at all.

In [ ]:
# ▶ DEMO
reading = ("2023-11-02", "Seattle", 8.4)
print(reading[1], len(reading))
try:
    reading[2] = 9.0
except TypeError as e:
    print("TypeError:", e)

print(type((5)).__name__, type((5,)).__name__, "<- the comma is what makes it")

In [ ]:
# ▶ DEMO — you have been unpacking these all afternoon
date, city, temp = reading
print(date, city, temp)

def low_high(values):
    return min(values), max(values)      # returning several things IS a tuple

lo, hi = low_high(temps)
print(lo, hi)

In [ ]:
# ▶ DEMO — a tuple can be a dict key. A list cannot.
movies = {("Aliens", 1986): 8.2, ("Aliens", 2016): 5.4}
print(movies[("Aliens", 1986)])

try:
    {["a", "b"]: 1}
except TypeError as e:
    print("TypeError:", e)

## 3.15 NumPy — a first look

Thirty minutes, and you meet it again tomorrow and Wednesday. **Three things to take away:** arrays are fast, **masks** filter, **axis** aggregates. That is what pandas is built on.

In [ ]:
# ▶ DEMO — why bother
import numpy as np

py_list = list(range(1_000_000))
np_arr  = np.arange(1_000_000)

%timeit [x * 2 for x in py_list]
%timeit np_arr * 2

In [ ]:
# ▶ DEMO — making one
a = np.array([1, 2, 3, 4, 5])
print(a.shape, a.dtype, a.mean())        # shape/dtype = attributes, mean() = method
print(np.arange(0, 10, 2), np.zeros(3), np.linspace(0, 1, 5))

print(np.array([1, 2, "three"]).dtype, "<- EVERYTHING became text. Check .dtype.")

In [ ]:
# ▶ DEMO — maths with no loop
temps_c = np.array([18.2, 21.7, 19.4, 25.1, 22.8, 30.3, 28.0])

print(temps_c * 9 / 5 + 32)
print(np.sqrt(temps_c).round(2))
print(temps_c.mean().round(2), temps_c.std().round(2))

## 3.16 Boolean Masks — the one to remember

In [ ]:
# 🔍 READ 3.16 — one answer, or seven?
#
#     temps_c = np.array([18.2, 21.7, 19.4, 25.1, 22.8, 30.3, 28.0])
#     print(temps_c > 25)
#
# Commit first.

prediction = "???"
check_read("3.16", prediction)

In [ ]:
# ▶ DEMO — and now index with it
mask = temps_c > 25
print(mask)
print(temps_c[mask])
print(temps_c[temps_c > 25], "<- usually written in one line")

print("\ncount:", mask.sum(), " fraction:", round(mask.mean(), 3))

In [ ]:
# ▶ DEMO — combining. The parentheses are REQUIRED.
mild = (temps_c > 19) & (temps_c < 26)
print(temps_c[mild])
print(temps_c[(temps_c < 19) | (temps_c > 28)])
print(temps_c[~mild])

```python
temps_c[temps_c > 25]          # today
df[df["temperature"] > 25]     # tomorrow
```

Same idea. Every filter you write in pandas for the rest of the week is a boolean mask.

## 3.17 Missing Values and Axis

In [ ]:
# ▶ DEMO — np.nan
a = np.array([1.0, np.nan, 3.0, np.nan, 5.0])
print(np.isnan(a))
print("mean   :", a.mean(), "<- one nan poisons everything")
print("nanmean:", np.nanmean(a))
print("np.nan == np.nan ->", np.nan == np.nan, "-- NEVER test with ==")

In [ ]:
# ▶ DEMO — axis is the dimension that DISAPPEARS
sales = np.array([[120,135,148,160],
                  [ 90, 95, 99,104],
                  [200,190,215,230],
                  [ 60, 75, 70, 88]])
regions = ["north","south","east","west"]

print("everything   :", sales.sum())
print("axis=0 (cols):", sales.sum(axis=0), sales.sum(axis=0).shape)
print("axis=1 (rows):", sales.sum(axis=1), sales.sum(axis=1).shape)
print("best region  :", regions[sales.sum(axis=1).argmax()])

## 3.18 Your Turn — 15 minutes

Same questions as 3.12, array version. Then one thing to read.

In [ ]:
# ▶ SETUP 3.18 — from cold. Blanks become np.nan.
import numpy as np
with open(PATH, newline="") as f:
    rows = list(csv.DictReader(f))

dates    = np.array([r["date"].strip() for r in rows])
cities   = np.array([r["city"].strip() for r in rows])
temps    = np.array([float(r["temperature (C)"]) if r["temperature (C)"].strip()
                     else np.nan for r in rows])

print(temps.shape, temps.dtype)
print(temps[:5])

In [ ]:
# ✍️ YOU TRY 3.18
# 1. n_missing     -- using np.isnan
# 2. pct_above_30  -- fraction of USABLE readings above 30 (a mask)
# 3. city_means    -- dict of city -> mean temperature, one mask per city
# 4. n_dupes       -- duplicate (date, city) pairs, using a set of tuples

n_missing    = 0    # TODO
pct_above_30 = 0.0  # TODO
city_means   = {}   # TODO
n_dupes      = 0    # TODO

print(n_missing, round(pct_above_30, 4))
print(city_means)
print(n_dupes)

In [ ]:
# ✅ CHECKPOINT 3.18
assert n_missing == 488, n_missing
assert abs(pct_above_30 - 0.2343) < 0.005, pct_above_30
assert set(city_means) == {"San Diego","Austin","New York","Phoenix","Seattle"}, sorted(city_means)
assert abs(city_means["Seattle"] - 19.3) < 0.15, city_means["Seattle"]
assert n_dupes == 3094, n_dupes

# and it must agree with what you computed by hand in 3.12
for c in city_means:
    assert abs(city_means[c] - averages[c]) < 0.2, (c, city_means[c], averages[c])
print("Correct -- and the array answers match your by-hand answers from 3.12.")

### 🔍 The reading question

Look at what you just computed.

```
Phoenix    20.4
New York   20.1
Austin     19.4
Seattle    19.3
San Diego  19.2
```

Every one of those numbers is **correct**. The code is right. The conclusion — *"Phoenix is the warmest of the five"* — is not something you should say out loud.

**Run the next cell, then answer the question below it.**

In [ ]:
# ▶ DEMO — is that 1.2 C spread bigger than chance?
# Shuffle the city labels at random and recompute. If real spreads look like
# shuffled spreads, the city column is telling us nothing.
import random
rng = random.Random(0)
usable_mask = ~np.isnan(temps)
vals   = temps[usable_mask]
labels = list(cities[usable_mask])

real = max(city_means.values()) - min(city_means.values())

spreads = []
for _ in range(300):
    rng.shuffle(labels)
    lab = np.array(labels)
    ms = [vals[lab == c].mean() for c in set(labels) if c and (lab == c).sum() > 50]
    spreads.append(max(ms) - min(ms))

print(f"observed spread between city means : {real:.2f} C")
print(f"average spread from SHUFFLED labels: {np.mean(spreads):.2f} C")
print(f"shuffles at least as big as ours   : {np.mean([s >= real for s in spreads]):.0%}")

**Write your answer here** (double-click to edit this cell):

> 1. What does the shuffle test tell you about the `city` column?
>
> 2. Check the months too — `date[:7]`. Every month averages between 18.7 and 21.0 °C. What does that tell you about this file?
>
> 3. You computed everything correctly. Write one sentence you *would* be willing to say out loud about this dataset.

*(your answer)*

---

**Model answer**

1. Shuffling the city labels produces a spread this large about a fifth of the time. The observed difference between cities is indistinguishable from chance — the `city` column carries no information about temperature.

2. There is no seasonality either. A real dataset of five US cities across a year would show Phoenix far hotter than Seattle, and July far hotter than January. This shows neither.

3. *"This file is synthetic — temperatures were drawn at random and the city and date labels attached afterwards. Any per-city or per-month finding from it is an artefact, and it should not be used to say anything about weather."*

#### That is the lesson of the whole afternoon.

Correct code, correct arithmetic, plausible-looking output — and a conclusion that would have been wrong. No error was raised. Nothing warned you.

**Somebody had to look.** That is the part that has not been automated, and it is what the rest of this week is about.

## Wrap

You can now:

- Store values, convert between text and numbers, and know when conversion fails
- Build conditions and branch on them in the right order
- Walk data six ways — `for`, `range`, `enumerate`, `zip`, `while`, and the early exits
- Choose between a list, dictionary, set and tuple, and count and group with them
- Filter with a **mask** and aggregate along an **axis**
- Read a traceback — and read a cell that has no traceback and is still wrong

**Next:** pandas ETL with Emin. Same file, same problems, far less typing.

**Tomorrow:** comprehensions, functions, and error handling — the tools for writing code you can still read next week.